# Collaborative Filtering Model

This notebook builds and tests a collaborative filtering recommendation system using user rating patterns and Truncated Singular Value Decomposition.

The model follows these steps:

1. Load the MovieLens datasets.
2. Create the user-item matrix.
3. Normalize ratings by user.
4. Train the Truncated SVD model.
5. Reconstruct predicted ratings.
6. Generate personalized movie recommendations.
7. Inspect model behavior and results.

In [10]:
import sys
from pathlib import Path

import pandas as pd

In [11]:
sys.path.append("../src")

from collaborative_filtering import (
    create_user_item_matrix,
    normalize_user_rating,
    train_svd_model,
    predict_ratings,
    recommend_movies_for_user
)

In [12]:
DATA_PATH = Path("../data/raw")

In [13]:
ratings = pd.read_csv(DATA_PATH / "ratings.csv")
movies = pd.read_csv(DATA_PATH / "movies.csv")

In [14]:
user_item_matrix = create_user_item_matrix(ratings)

user_item_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
user_item_matrix.shape

(610, 9724)

In [ ]:
normalized_matrix, user_means = normalize_user_rating(
    user_item_matrix
)

normalized_matrix.head()

In [18]:
svd, user_factors, item_factors = train_svd_model(
    normalized_matrix,
    n_components=50
)

In [21]:
predicted_ratings = predict_ratings(
    user_factors=user_factors,
    item_factors=item_factors,
    user_means=user_means,
    user_item_matrix=user_item_matrix
)

predicted_ratings.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.352589,4.277628,4.338038,4.354390,4.341245,4.075129,4.182902,4.352091,4.383275,4.227027,...,4.366369,4.366343,4.366394,4.366394,4.366369,4.366394,4.366369,4.366369,4.366369,4.365413
2,3.936643,3.936795,3.942364,3.952944,3.955821,3.937100,3.927488,3.951571,3.952765,3.979641,...,3.948219,3.948079,3.948358,3.948358,3.948219,3.948358,3.948219,3.948219,3.948219,3.947876
3,2.439763,2.463286,2.389917,2.445750,2.461446,2.403453,2.453444,2.457896,2.427255,2.449779,...,2.435857,2.435759,2.435955,2.435955,2.435857,2.435955,2.435857,2.435857,2.435857,2.436236
4,3.498435,3.632830,3.514086,3.490894,3.506224,3.591726,3.585116,3.511527,3.546550,3.466259,...,3.556745,3.559642,3.553847,3.553847,3.556745,3.553847,3.556745,3.556745,3.556745,3.551004
5,3.813666,3.616240,3.629705,3.616007,3.603547,3.745175,3.617735,3.620280,3.622036,3.587167,...,3.636527,3.636925,3.636129,3.636129,3.636527,3.636129,3.636527,3.636527,3.636527,3.637025


In [22]:
USER_ID = 1

In [23]:
recommendations = recommend_movies_for_user(
    user_id=USER_ID,
    predicted_ratings=predicted_ratings,
    user_item_matrix=user_item_matrix,
    movies=movies,
    top_n=10
)

recommendations

,movieId,title,genres,predicted_rating
0,1246,Dead Poets Society (1989),Drama,4.608672
1,4011,Snatch (2000),Comedy|Crime|Thriller,4.587101
2,1035,"Sound of Music, The (1965)",Musical|Romance,4.576907
3,2081,"Little Mermaid, The (1989)",Animation|Children|Comedy|Musical|Romance,4.574988
4,2762,"Sixth Sense, The (1999)",Drama|Horror|Mystery,4.574478
5,904,Rear Window (1954),Mystery|Thriller,4.572992
6,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,4.572973
7,2804,"Christmas Story, A (1983)",Children|Comedy,4.571532
8,4306,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,4.569328
9,1148,Wallace & Gromit: The Wrong Trousers (1993),Animation|Children|Comedy|Crime,4.568308


In [24]:
ratings.loc[
    ratings["userId"] == USER_ID
].merge(
    movies[["movieId","title","genres"]],
    on="movieId"
).sort_values(
    "rating",
    ascending=False
)[["title","genres","rating"]]

,title,genres,rating
231,M*A*S*H (a.k.a. MASH) (1970),Comedy|Drama|War,5.0
185,Excalibur (1981),Adventure|Fantasy,5.0
89,Indiana Jones and the Last Crusade (1989),Action|Adventure,5.0
90,Pink Floyd: The Wall (1982),Drama|Musical,5.0
190,From Russia with Love (1963),Action|Adventure|Thriller,5.0
...,...,...,...
170,"Mummy, The (1999)",Action|Adventure|Comedy|Fantasy|Horror|Thriller,2.0
143,Toys (1992),Comedy|Fantasy,2.0
148,I Still Know What You Did Last Summer (1998),Horror|Mystery|Thriller,2.0
152,Psycho (1998),Crime|Horror|Thriller,2.0


In [25]:
print(
    f"Explained variance: "
    f"{svd.explained_variance_ratio_.sum():.2%}"
)

Explained variance: 48.88%


In [26]:
print(user_factors.shape)
print(item_factors.shape)

(610, 50)
(50, 9724)
